<a href="https://colab.research.google.com/github/timraiswell/ai-engineer/blob/main/03-rag/04-why-rag-fails.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Why RAG Fails

**Goal:** Diagnose bad answers. Retrieval quality, not generation, is usually the bottleneck.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client; `requests`, `sentence-transformers`, `numpy` are used by this notebook.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [1]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git" requests sentence-transformers numpy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.4 MB/s eta 0:00:00


In [2]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

Groq client ready. MODEL = openai/gpt-oss-120b


## The thesis

When a RAG system gives a bad answer, the reflex is to blame the model and start rewriting the prompt. That reflex is usually wrong. In most broken RAG systems the model never saw the answer: retrieval failed, quietly, and generation did something fluent with the wrong material.

> **⭐ Key takeaway —** the diagnosis discipline is one rule: **when an answer is wrong, check retrieval first.** Look at what was actually in the prompt before you touch anything else.

This notebook builds a tiny labeled set, measures retrieval against it, and walks through the four ways RAG fails, each demonstrated live against our corpus, each with a programmatic detection and a cheapest fix.

## Setup: corpus, chunks, retrieval

Same ten-RFC corpus, section-aware chunks, and brute-force vector search from notebook 01 (cells repeated, since every notebook here is self-contained).


In [3]:
# Download the shared corpus: ten IETF RFCs, cached to data/rfc/.
# Every notebook in this repo that needs the corpus includes this cell --
# self-containment over DRY, so each notebook runs top-to-bottom on its own.
import os
import requests

RFC_NUMBERS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
RFC_TITLES = {
    791: 'IP', 793: 'TCP', 1035: 'DNS', 2616: 'HTTP/1.1', 4271: 'BGP',
    5321: 'SMTP', 6455: 'WebSocket', 6749: 'OAuth 2.0', 7540: 'HTTP/2',
    9110: 'HTTP Semantics',
}
DATA_DIR = 'data/rfc'
os.makedirs(DATA_DIR, exist_ok=True)

corpus = {}  # rfc number -> raw text
for num in RFC_NUMBERS:
    path = os.path.join(DATA_DIR, f'rfc{num}.txt')
    if not os.path.exists(path):  # cached: skip the download on re-runs
        resp = requests.get(f'https://www.rfc-editor.org/rfc/rfc{num}.txt', timeout=30)
        resp.raise_for_status()
        with open(path, 'w') as f:
            f.write(resp.text)
    with open(path) as f:
        corpus[num] = f.read()

for num in RFC_NUMBERS:
    print(f'RFC {num:>4}  {RFC_TITLES[num]:<15} {len(corpus[num]):>9,} chars')

RFC  791  IP                 94,892 chars
RFC  793  TCP               172,710 chars
RFC 1035  DNS               122,549 chars
RFC 2616  HTTP/1.1          422,279 chars
RFC 4271  BGP               222,702 chars
RFC 5321  SMTP              225,929 chars
RFC 6455  WebSocket         162,067 chars
RFC 6749  OAuth 2.0         163,498 chars
RFC 7540  HTTP/2            209,580 chars
RFC 9110  HTTP Semantics    502,907 chars


In [4]:
# Compact copy of the cleaning + section-aware chunking. Notebook 03 (chunking)
# is where this is built up and its tradeoffs explained; here it's just a given.
import re

def clean_rfc(text):
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.match(r'^.*\[Page \d+\]\s*$', l)        # page footers
             and not re.match(r'^RFC \d+\s+.*\S+ \d{4}\s*$', l)]  # page headers
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines)).strip()

HEADING_RE = re.compile(r'^\d+(?:\.\d+)*\.?\s+\S', re.MULTILINE)

def chunk_paragraphs(text, max_chars=2400):
    paras = [p for p in text.split('\n\n') if p.strip()]
    chunks, cur = [], ''
    for p in paras:
        if cur and len(cur) + len(p) + 2 > max_chars:
            chunks.append(cur)
            cur = p
        else:
            cur = cur + '\n\n' + p if cur else p
    if cur:
        chunks.append(cur)
    return chunks

def chunk_sections(text, max_chars=2400):
    starts = [m.start() for m in HEADING_RE.finditer(text)]
    if not starts:
        return chunk_paragraphs(text, max_chars)
    chunks = [text[:starts[0]].strip()] if text[:starts[0]].strip() else []
    for a, b in zip(starts, starts[1:] + [len(text)]):
        sec = text[a:b].strip()
        if len(sec) <= max_chars:
            chunks.append(sec)
        else:  # long section: paragraph-pack it, carrying the heading along
            heading = sec.split('\n', 1)[0]
            for sub in chunk_paragraphs(sec, max_chars):
                chunks.append(sub if sub.startswith(heading) else heading + '\n' + sub)
    return [c for c in chunks if c]

chunk_texts, chunk_meta = [], []
for num in RFC_NUMBERS:
    for c in chunk_sections(clean_rfc(corpus[num])):
        chunk_texts.append(c)
        chunk_meta.append({'rfc': num, 'title': RFC_TITLES[num]})
print(f'{len(chunk_texts)} chunks across {len(RFC_NUMBERS)} RFCs')

1588 chunks across 10 RFCs


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunk_texts, normalize_embeddings=True,
                             show_progress_bar=True)

def search(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = embeddings @ q
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), int(i)) for i in top]

def answer(question, k=5, system=None):
    hits = search(question, k)
    context = '\n\n'.join(
        f'[{n + 1}] (RFC {chunk_meta[i]["rfc"]}, {chunk_meta[i]["title"]})\n{chunk_texts[i][:1500]}'
        for n, (_, i) in enumerate(hits)
    )
    prompt = (
        'Answer the question using the numbered passages below. '
        'Cite passages like [1].\n\n'
        f'{context}\n\nQuestion: {question}'
    )
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=1024, messages=messages,
    )
    return resp.choices[0].message.content

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

## A tiny labeled set

Ten questions, each with the RFC that contains the answer and a phrase that identifies the answering chunk. This took about twenty minutes to write by hand, and it converts "retrieval feels okay" into a number. Labels beat vibes at any scale, even n=10.


In [6]:
labeled = [
    {'q': 'What HTTP status code means the request body is larger than the server will process?',
     'rfc': 9110, 'phrase': 'Content Too Large'},
    {'q': 'Which HTTP status code says the service is temporarily overloaded and the client may retry later?',
     'rfc': 9110, 'phrase': '503'},
    {'q': 'What port does SMTP use for mail transfer between servers?',
     'rfc': 5321, 'phrase': 'port 25'},
    {'q': 'What handshake establishes a TCP connection?',
     'rfc': 793, 'phrase': 'three-way handshake'},
    {'q': 'What GUID does a WebSocket server concatenate with the client key to compute Sec-WebSocket-Accept?',
     'rfc': 6455, 'phrase': '258EAFA5-E914-47DA-95CA-C5AB0DC85B11'},
    {'q': 'In OAuth 2.0, which grant type exchanges an authorization code for an access token?',
     'rfc': 6749, 'phrase': 'authorization_code'},
    {'q': 'Which BGP message type advertises routes to a peer?',
     'rfc': 4271, 'phrase': 'UPDATE message'},
    {'q': 'What does the Time to Live field in the IP header do?',
     'rfc': 791, 'phrase': 'time to live'},
    {'q': 'How does HTTP/2 carry many requests concurrently over one connection?',
     'rfc': 7540, 'phrase': 'multiplexing'},
    {'q': 'What DNS record type holds a host address?',
     'rfc': 1035, 'phrase': 'host address'},
]

def rank_of_correct(item, max_k=50):
    """Rank (1-based) of the first chunk from the right RFC containing the
    answer phrase, within the top max_k -- or None if it never shows up."""
    for rank, (_, i) in enumerate(search(item['q'], max_k), 1):
        if (chunk_meta[i]['rfc'] == item['rfc']
                and item['phrase'].lower() in chunk_texts[i].lower()):
            return rank
    return None

def hit_rate(k):
    ranks = [rank_of_correct(it) for it in labeled]
    return sum(1 for r in ranks if r is not None and r <= k) / len(labeled)

for it in labeled:
    r = rank_of_correct(it)
    print(f'rank {str(r):>4}  RFC {it["rfc"]:>4}  {it["q"][:64]}')
print()
print(f'hit rate @5:  {hit_rate(5):.0%}')
print(f'hit rate @20: {hit_rate(20):.0%}')

rank    1  RFC 9110  What HTTP status code means the request body is larger than the 
rank    2  RFC 9110  Which HTTP status code says the service is temporarily overloade
rank    2  RFC 5321  What port does SMTP use for mail transfer between servers?
rank    2  RFC  793  What handshake establishes a TCP connection?
rank    6  RFC 6455  What GUID does a WebSocket server concatenate with the client ke
rank   25  RFC 6749  In OAuth 2.0, which grant type exchanges an authorization code f
rank    1  RFC 4271  Which BGP message type advertises routes to a peer?
rank    3  RFC  791  What does the Time to Live field in the IP header do?
rank    2  RFC 7540  How does HTTP/2 carry many requests concurrently over one connec
rank    3  RFC 1035  What DNS record type holds a host address?

hit rate @5:  80%
hit rate @20: 90%


Run this and read the rank column before the summary numbers. Every question lands in one of a few buckets: rank 1–5 (retrieval fine), rank 6–50 (retrieved but buried), or `None` (never retrieved). Those buckets map directly onto the failure classes below, and notice you learned all of this *without generating a single answer*. Retrieval metrics are cheap; answer evals cost API calls. Always measure the cheap layer first.

## Failure class 1: the answer never got retrieved

**Symptom:** rank is `None` (or worse than anything you'd put in a prompt), yet the answer *is* in the corpus.

**Detect programmatically:** `rank_of_correct(item) is None` while a corpus grep for the phrase succeeds. That pair of facts is the fingerprint. If the grep also fails, you're in class 3 instead.


In [7]:
def in_corpus(phrase):
    return any(phrase.lower() in t.lower() for t in chunk_texts)

for it in labeled:
    r = rank_of_correct(it)
    if r is None or r > 5:
        print(f'rank {str(r):>4}  in_corpus={in_corpus(it["phrase"])}  {it["q"][:60]}')

# The two usual causes, on demand:
# (a) vocabulary mismatch -- the query and the answering text share no words
print()
print('vocabulary-mismatch probe:')
for _, i in search('how do I stop mail from bouncing between servers forever?', 3):
    print(f'   RFC {chunk_meta[i]["rfc"]:>4}  {chunk_texts[i].strip().split(chr(10))[0][:64]}')
print('   (answer: RFC 5321 sec 6.3 Loop Detection -- counting Received: header')
print('    fields. The query says "bouncing forever"; the RFC never does.)')

rank    6  in_corpus=True  What GUID does a WebSocket server concatenate with the clien
rank   25  in_corpus=True  In OAuth 2.0, which grant type exchanges an authorization co

vocabulary-mismatch probe:
   RFC 5321  4.5.4.1.  Sending Strategy
   RFC 5321  4.5.4.  Retry Strategies
   RFC 5321  4.5.4.1.  Sending Strategy
   (answer: RFC 5321 sec 6.3 Loop Detection -- counting Received: header
    fields. The query says "bouncing forever"; the RFC never does.)


Run this and note which labeled questions (if any) fall in this class on your index; results vary a little with the embedding model version. The two usual causes:

- **Chunking split the answer.** The identifying phrase and the substance ended up in different chunks, so no single chunk matches the query well. Notebook 03's boundary demo is exactly this failure being born.
- **Vocabulary mismatch.** The query says "stop mail bouncing forever," the RFC says "detecting loops" by "counting Received: header fields." Embeddings close some of that gap but not all of it, and keyword search closes none.

**Cheapest fix:** rechunk (bigger sections, or heading-in-every-chunk like our section chunker, notebook 03) for the first cause; hybrid retrieval (notebook 02) for the second. Both are index-side fixes, no prompt engineering involved.

## Failure class 2: retrieved, but drowned

**Symptom:** the right chunk exists in the top 20 but not the top 5, say rank 8. You only pass 5 chunks to the model, so it answers from the four-and-a-half wrong ones. Even when you pass more, models attend most reliably to the top of the context; an answer buried under seven near-misses is easy to ignore.

**Detect programmatically:** hit@20 true, hit@5 false. The gap between those two numbers *is* this failure class, measured.


In [8]:
drowned = [it for it in labeled
           if (r := rank_of_correct(it)) is not None and 5 < r <= 20]

print(f'hit@5 = {hit_rate(5):.0%}, hit@20 = {hit_rate(20):.0%}')
print(f'-> {len(drowned)} question(s) are retrieved-but-drowned:')
for it in drowned:
    print(f'   rank {rank_of_correct(it)}: {it["q"][:64]}')
    print(answer(it['q'], k=5)[:300])   # watch it answer without the right chunk
    print()

hit@5 = 80%, hit@20 = 90%
-> 1 question(s) are retrieved-but-drowned:
   rank 6: What GUID does a WebSocket server concatenate with the client ke
The server must append the fixed GUID  

**`258EAFA5‑E914‑47DA‑95CA‑C5AB0DC85B11`**  

to the client’s Sec‑WebSocket‑Key before taking the SHA‑1 hash to form the Sec‑WebSocket‑Accept value【3】.



Run this and read the generated answers for the drowned questions: the model does its best with the five chunks it got, often producing something plausible-sounding that cites the wrong section, which is far more dangerous than an obvious failure.

**Cheapest fix:** reranking (notebook 02). The candidate pool already contains the answer; you just need a better sort of the top 20 before you cut to 5. Widening k from 5 to 20 also "works" but spends 4× the context tokens on every query to fix a sorting problem, and the reranker is cheaper at scale.

## Failure class 3: not in the corpus at all, the silent one

**Symptom:** none, and that's the problem. Ask about something the corpus doesn't cover, and retrieval dutifully returns the nearest chunks (pitfall 3 from notebook 01: nearest-neighbor always returns *something*), and the model, which knows the answer from pretraining, writes a confident, correct-sounding response that your corpus never supported. Users learn to trust it. Then it does the same thing on a question where its pretraining is wrong or stale, with the same confident tone.

PKCE is RFC 7636 (2015), an OAuth 2.0 extension published *after* RFC 6749 and absent from our ten (the string `PKCE` appears nowhere in the corpus; check with `any('pkce' in t.lower() for t in chunk_texts)`). The model, of course, knows it cold from pretraining. Watch:


In [9]:
q = 'How does PKCE protect the OAuth authorization code flow?'

print('top-3 retrieval scores (nothing relevant exists):')
for score, i in search(q, 3):
    print(f'  {score:.3f}  RFC {chunk_meta[i]["rfc"]:>4} ({chunk_meta[i]["title"]})')
print()
print('--- naive RAG answer:')
print(answer(q)[:600])

top-3 retrieval scores (nothing relevant exists):
  0.563  RFC 6749 (OAuth 2.0)
  0.536  RFC 6749 (OAuth 2.0)
  0.534  RFC 6749 (OAuth 2.0)

--- naive RAG answer:
PKCE (Proof Key for Code Exchange) adds a **cryptographic verifier** to the standard Authorization‑Code flow, turning the otherwise‑plain‑text, bearer‑only code into a value that can only be redeemed by the party that originally started the flow.  

1. **The authorization code is a short‑lived, single‑use bearer credential.**  
   Because the code travels through the user‑agent it can be captured (e.g., via browser history or referrer headers) — the spec therefore requires the client’s redirection endpoint to be protected with TLS and the client to authenticate when exchanging the code [1].  



Run this and note two things. First, retrieval looks *healthy*: it returns OAuth chunks with respectable scores, because PKCE questions are semantically close to RFC 6749 material, so the scores don't flag anything. Second, the answer is probably *correct* (the model knows PKCE from pretraining) and it cites passages that don't actually support it, or waves at them vaguely. Correct-but-unsupported is still a failure: you built RAG precisely so answers would be grounded in *your* documents, and this answer is grounded in nothing you control. It will fail the same confident way on a question where pretraining is wrong or stale.

**Detect programmatically:** hard, and that's the point. Practical detectors: a corpus grep for key entities from the question ("does the literal string `PKCE` appear anywhere?"), a citation-support check (does the cited passage actually contain the claim?), or an eval set that includes known-unanswerable questions and expects "I don't know."

**Cheapest fix:** tell the model refusal is an acceptable answer, and make the instruction concrete:


In [16]:
GROUNDED = ('Answer ONLY from the provided passages. If the passages do not '
            'contain the information needed, reply exactly: "The corpus does '
            'not cover this!" Do not use outside knowledge, even if you know '
            'the answer.')

print(answer(q, system=GROUNDED)[:400])
print()
# And confirm the instruction doesn't break answerable questions:
print(answer('What port does SMTP use for mail transfer between servers?',
             system=GROUNDED)[:400])

The corpus does not cover this!

SMTP uses the standard SMTP port 25 for server‑to‑server mail transfer【2】.


Run both and check: the PKCE question should now be declined, and the SMTP question should still be answered. That second check matters. An over-aggressive refusal instruction that makes the model decline answerable questions is a regression, and you'd only catch it by re-running the answerable set. (You're starting to feel why section 04 exists.)

## Failure class 4: retrieved correctly, but the question is multi-hop

**Symptom:** the question needs facts from *two documents*, and one query's top-k clusters around whichever document the query's wording favors. Each individual chunk was retrieved "correctly." There's just no single chunk, or single neighborhood, that covers the comparison.


In [ ]:
q = ('How do SMTP and HTTP each tell a client that a failure is temporary '
     'and the request should be retried later?')

print('single-query top-5 -- count RFCs represented:')
for score, i in search(q, 5):
    print(f'  {score:.3f}  RFC {chunk_meta[i]["rfc"]:>4} ({chunk_meta[i]["title"]})')

Run this and count the RFCs in the top 5: typically one protocol dominates and the other barely appears, so an answer generated from these hits covers half the question well and improvises the other half.

**Detect programmatically:** questions naming two entities whose top-k is dominated by one source, e.g. `len({meta['rfc'] for hits}) == 1` when the question mentions two protocols. Crude, but it flags the class.

**Cheapest fix:** query decomposition. Split the question into sub-queries (by hand or with one cheap model call), retrieve for each, and merge before generating:


In [ ]:
sub_queries = [
    'How does SMTP indicate a temporary failure that the client should retry?',
    'How does HTTP indicate a temporary failure that the client should retry?',
]

merged, seen = [], set()
for sq in sub_queries:
    for score, i in search(sq, 3):
        if i not in seen:
            seen.add(i)
            merged.append((score, i))

print('merged hits -- both protocols now represented:')
for score, i in merged:
    print(f'  {score:.3f}  RFC {chunk_meta[i]["rfc"]:>4} ({chunk_meta[i]["title"]})')

context = '\n\n'.join(
    f'[{n + 1}] (RFC {chunk_meta[i]["rfc"]}, {chunk_meta[i]["title"]})\n{chunk_texts[i][:1500]}'
    for n, (_, i) in enumerate(merged)
)
resp = client.chat.completions.create(
    model=MODEL, max_tokens=1024,
    messages=[{'role': 'user', 'content':
               f'Answer using only the passages, citing like [1].\n\n{context}\n\nQuestion: {q}'}],
)
print()
print(resp.choices[0].message.content[:700])

Run this and compare the merged hit list with the single-query one: both RFCs represented, and the generated comparison can now cite each side. Decomposition is the cheapest multi-hop fix; agentic retrieval (letting the model issue follow-up queries itself, section 05 territory) is the expensive general version of the same idea.

## The diagnosis card

When an answer is wrong, work this list in order. It's ordered by cheapness of both the check and the fix:

| # | Class | Detect | Cheapest fix |
|---|---|---|---|
| 1 | Answer not retrieved | rank is `None`, corpus grep succeeds | rechunk / hybrid retrieval |
| 2 | Retrieved but drowned | hit@20 ✓, hit@5 ✗ | rerank the candidate pool |
| 3 | Not in corpus | corpus grep fails; citation doesn't support claim | say-I-don't-know instructions + unanswerable evals |
| 4 | Multi-hop | top-k dominated by one source on a two-entity question | query decomposition |

And a closing honesty check: hit-rate on 10 hand-written questions is a **proto-eval**. It converted vibes into numbers and told us where the pipeline breaks, but n=10 can't distinguish a 78% system from an 85% one, it doesn't score answer *quality*, and we eyeballed the generations instead of grading them. Section 04 takes exactly this artifact (questions, labels, hit-rate) and makes it rigorous: bigger golden sets, LLM-as-judge for answer quality, and regression evals so the number is a gate instead of a snapshot.

## Where the frameworks come in: LangChain & LlamaIndex

You built retrieval, hybrid + reranking, chunking, and failure diagnosis from raw parts: embeddings in numpy, a `search()` function, prompts you can read. That was on purpose. **RAG is a pattern, not a library**, and now you can see exactly what any RAG framework is doing under its abstractions.

**LangChain** and **LlamaIndex** are the two you'll see on job posts. At their core they wire the *same* retrieve → augment → generate loop you built, and add:

- **Connectors:** loaders for PDFs, Notion, Slack, SQL, web; dozens of vector stores (Pinecone, Chroma, pgvector) behind one interface. This is the real time-saver, since you stop writing ingestion glue.
- **Composable pipelines:** retriever → reranker → prompt → LLM as interchangeable components (LangChain's LCEL, LlamaIndex's query engines) instead of hand-wired functions.
- **Prebuilt strategies:** the multi-query and sub-question retrieval you just wrote by hand ship as `MultiQueryRetriever` / sub-question query engines; parent-document and sentence-window chunking come ready-made.
- **Agentic RAG:** routing a query to the right index, or letting an agent (section 05) decide *when* to retrieve, wired for you.

> **⭐ Key takeaway —** a framework changes *how you assemble* RAG, not *what RAG is*. Every failure mode from this section (nothing retrieved, retrieved-but-drowned, not-in-corpus, multi-hop) happens **inside** LangChain and LlamaIndex too, and they won't diagnose it for you. You debug them with exactly the retrieval-quality lens you built here.

**When to reach for one:** when the connector/vector-store breadth saves real glue code, or your team already standardized on it. **When not:** a focused production pipeline is often *clearer* as the ~40 lines you wrote than as a stack of abstractions you'd have to reverse-engineer when retrieval quality drops. Either way, you'll evaluate the framework well, because you know what it's abstracting.

> **🔵 Interview signal —** "I've used LangChain, but I can also explain what it does under the hood and when raw retrieval is the better call" is a far stronger answer than "we used LangChain" *or* "I'd never use a framework." Fluency plus judgment.

## Exercises

1. **Grow the labeled set.** Add 5 questions of your own, including at least one you *expect* to fail (an exact identifier, a paraphrase with no keyword overlap) and one that's unanswerable from the corpus. Recompute hit@5 and hit@20. Did the failures land in the classes you predicted?
2. **Hybrid vs vector on the labeled set.** Port `bm25_search` and `rrf` from notebook 02 into this notebook and compute hit@5 for vector-only vs hybrid on all labeled questions. You now have the first real evidence for whether hybrid earns its cost on this corpus: one number, not a hunch.
3. **Citation-support checker.** Write `supports(claim_answer, cited_chunks)` using one model call: does the cited text actually contain the answer given? Run it over the class-3 PKCE answer and over three answerable questions. This is the seed of a groundedness eval.
4. **Automatic failure triage.** Write `triage(item)` that returns `'ok'`, `'not_retrieved'`, `'drowned'`, or `'not_in_corpus'` using `rank_of_correct` and `in_corpus`. Run it over the full labeled set and print a class histogram. You've built the diagnosis card as a function, which is exactly the shape section 04 expects from you.
